# Project 4: SSC History — World War I Answer Grader

We evaluate an **LLM judge**: can it apply a marking rubric consistently?

Five practice questions × five fictional students = 25 answers.
Each question is worth 10 marks, awarded across five dimensions of 2 marks each.
Jack is casual but informed; Jill is meticulous; Jane is poorly prepared;
John is average; Jake is unprepared. The judge never sees these identities.

Reference grades were **simulated by an AI assistant and require human review**.
They demonstrate MAE calculation; they are not genuine human-validated ground truth.
These are short-answer learning exercises, not an official SSC paper.

Use the shared root `.venv` kernel. Run cells individually from the repository
root or Project 4 folder. No new `uv init` or vector database is needed.
Restart the kernel after editing helpers or switching from another project's
`models`/`tools` modules.


## 1. Set the project path
Keep the notebook focused on the experiment; supporting code lives in the project folders.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name != "4_ssc_wwi_grader":
    project_root = project_root / "4_ssc_wwi_grader"
assert project_root.is_dir(), "Run from the repository root or Project 4 folder."
sys.path.insert(0, str(project_root))

from tools.results import read_json, save_run, print_report
from tools.evaluation import validate_dataset, compare_grades


## 2. Load the questions, answers, and reference grades

Questions contain marking anchors and an ideal factual answer.
The separate reference-grade file contains marks already assigned to each
student answer. It is used only after model grading, never in the judge prompt.


In [2]:
questions = read_json(project_root / "docs/goldends/questions.json")
students = read_json(project_root / "docs/dataset/students.json")
answers = read_json(project_root / "docs/dataset/student_answers.json")
references = read_json(project_root / "docs/goldends/reference_grades.json")
reference_metadata = read_json(project_root / "docs/goldends/reference_metadata.json")

validate_dataset(questions, students, answers, references)
print(f"Questions: {len(questions)} | Students: {len(students)} | Answers: {len(answers)}")
print(f"Reference status: {reference_metadata['status']}")
for question in questions:
    print(f"{question['id']} ({question['max_marks']} marks): {question['question']}")


Questions: 5 | Students: 5 | Answers: 25
Reference status: simulated_pending_human_review
q001 (10 marks): Explain the long-term causes of World War I and how the assassination in June 1914 escalated into a wider war by August 1914.
q002 (10 marks): Identify the main opposing camps in World War I. Distinguish the countries fighting in 1914 from later entrants, including Italy, the Ottoman Empire, Bulgaria and the United States.
q003 (10 marks): What was trench warfare on the Western Front? Explain why it produced prolonged stalemate, and mention the conditions soldiers faced.
q004 (10 marks): Why did the United States enter World War I in 1917, how did it contribute, and what is the difference between the Armistice of 11 November 1918 and the later peace treaties?
q005 (10 marks): When and with whom was the Treaty of Versailles signed? Describe its main terms for Germany and explain its significance without treating World War II as an inevitable result of this treaty alone.


## 3. Understand one worked reference

Inspect the rubric, student answer, and simulated reference marks. Each dimension
earns 0, 1, or 2; the total is their sum. Informal wording is not penalized if
its meaning is accurate. Missing or false facts do not earn assumed credit.


In [3]:
example = answers[0]
question = next(row for row in questions if row["id"] == example["question_id"])
reference = next(row for row in references if row["answer_id"] == example["id"])

print(example["id"])
print(example["answer"])
print("\nRubric:")
for dimension in question["dimensions"]:
    print(f"{dimension['id']}: {dimension['criterion']} ({dimension['max_marks']} marks)")
print("\nSimulated reference:")
for row in reference["dimension_marks"]:
    print(f"{row['dimension_id']}: {row['marks']} — {row['reason']}")
print(f"Total: {reference['total_marks']} / {question['max_marks']}")


jack_q001
Europe was already a pretty tense place. Countries competed for colonies, nationalism encouraged rivalry, and the arms race meant everyone had powerful armies ready. Alliances made a local fight everyone's problem. The spark was Franz Ferdinand's assassination in Sarajevo in June 1914. Austria-Hungary then declared war on Serbia. Russia backed Serbia, Germany backed Austria-Hungary, and Germany fought Russia and France. Britain joined after Germany invaded Belgium. So the assassination set off tensions that had been building for years.

Rubric:
d1: Nationalism and imperial rivalry (2 marks)
d2: Militarism and rival alliances (2 marks)
d3: The immediate trigger (2 marks)
d4: Austria-Hungary and Serbia (2 marks)
d5: Escalation into a European war (2 marks)

Simulated reference:
d1: 2 — Connects competition for colonies and nationalism to European rivalry.
d2: 2 — Explains armed preparedness and how alliances widened a local conflict.
d3: 1 — Names Franz Ferdinand, Sarajevo and 

## 4. Connect the LLM judge

This needs `GROQ_API_KEY` in the shared root `.env`.
The model returns per-dimension marks and brief reasons as JSON. Python checks
the response and adds the marks. No custom retry or error handler is used.

The system prompt is in `prompts/grader.txt`; model settings are in `models/`.


In [4]:
from models.llm import load_model
from tools.grading import grade_answer, grade_all

model = load_model()
prompt_text = (project_root / "prompts/grader.txt").read_text(encoding="utf-8")


## 5. Try one answer — optional

This makes one model call. The helper receives only anonymous answer text,
the question, reference answer, and rubric. Names, personas, and simulated
reference marks are absent from the request.


In [5]:
trial_grade = grade_answer(model, question, example["answer"], prompt_text)
for row in trial_grade["dimension_marks"]:
    print(f"{row['dimension_id']}: {row['marks']} — {row['reason']}")
print(f"Model total: {trial_grade['total_marks']} / {question['max_marks']}")


d1: 2 — Mentions both nationalism and imperial competition and explains their role in tension.
d2: 2 — Explains militarism via arms race and notes the impact of alliance system.
d3: 1 — Correctly identifies assassination in Sarajevo but does not give exact date.
d4: 1 — Notes Austria-Hungary's declaration of war on Serbia but omits the specific date.
d5: 1 — Links German invasion of Belgium and British entry, but does not fully describe Russian mobilization and full escalation chain.
Model total: 7 / 10


## 6. Grade all 25 answers

This makes 25 model calls and saves a new timestamped run only after they all
succeed. Every answer uses the same rubric and prompt. Prior runs remain intact.

API calls may have a cost or rate limits. If a cell fails, inspect the error;
no new complete grade file is saved. Skip this and the model cells when using
an existing saved run.


In [6]:
model_run_path = grade_all(model, questions, answers, prompt_text, label="baseline")
print(f"Model grades saved: {model_run_path.name}")


Graded jack_q001
Graded jack_q002
Graded jack_q003
Graded jack_q004
Graded jack_q005
Graded jill_q001
Graded jill_q002
Graded jill_q003
Graded jill_q004
Graded jill_q005
Graded jane_q001
Graded jane_q002
Graded jane_q003
Graded jane_q004
Graded jane_q005
Graded john_q001
Graded john_q002
Graded john_q003
Graded john_q004
Graded john_q005
Graded jake_q001
Graded jake_q002
Graded jake_q003
Graded jake_q004
Graded jake_q005
Model grades saved: grades_20261015_090422_064100.json


## 7. Select model grades

This lists saved runs and selects the latest. Use an explicit path to select an
older run. To work offline, run steps 1 and 2, then continue here; loading saved
grades does not require an API key or a model connection.


In [7]:
saved_runs = sorted((project_root / "docs/evaluatedds").glob("grades_*.json"))
assert saved_runs, "Run the judge on all answers first."
for path in saved_runs:
    print(path.name)

model_run_path = saved_runs[-1]
model_run = read_json(model_run_path)
print(f"Selected run: {model_run['run_id']}")


grades_20261015_090422_064100.json
Selected run: grades_20261015_090422_064100


## 8. Compare with the simulated reference marks

`Absolute error = abs(model mark - reference mark)`

`MAE = sum of the 25 absolute errors / 25`

For example, reference 8 and model 6 gives an error of 2 marks.
Lower MAE means closer agreement. A practice target of MAE ≤ 1 mark is defined
before running the experiment; it is not proof that the grader is reliable.

The helper checks version fingerprints, complete answer IDs, legal marks, and
totals before comparison. Results remain labelled as a comparison with simulated
references until real human review has been completed.


In [10]:
report = compare_grades(
    questions, students, answers, references, model_run, reference_metadata
)
print_report(report,students)
report_path = save_run(project_root / "docs/evalresults", "evaluation", report)
print(f"Report saved: {report_path.name}")


Reference status: simulated_pending_human_review
Answers compared: 25
MAE: 0.20 marks per answer
Practice target <= 1.00: True

Student | Reference total | Model total | MAE
Jack | 43 | 43 | 0.00
Jill | 50 | 50 | 0.00
Jane | 16 | 16 | 0.40
John | 33 | 32 | 0.60
Jake | 0 | 0 | 0.00

Answer | Reference | Model | Absolute error
jack_q001 | 7 | 7 | 0
jack_q002 | 9 | 9 | 0
jack_q003 | 10 | 10 | 0
jack_q004 | 9 | 9 | 0
jack_q005 | 8 | 8 | 0
jill_q001 | 10 | 10 | 0
jill_q002 | 10 | 10 | 0
jill_q003 | 10 | 10 | 0
jill_q004 | 10 | 10 | 0
jill_q005 | 10 | 10 | 0
jane_q001 | 5 | 6 | 1
jane_q002 | 5 | 5 | 0
jane_q003 | 4 | 3 | 1
jane_q004 | 1 | 1 | 0
jane_q005 | 1 | 1 | 0
john_q001 | 7 | 6 | 1
john_q002 | 8 | 8 | 0
john_q003 | 7 | 6 | 1
john_q004 | 6 | 7 | 1
john_q005 | 5 | 5 | 0
jake_q001 | 0 | 0 | 0
jake_q002 | 0 | 0 | 0
jake_q003 | 0 | 0 | 0
jake_q004 | 0 | 0 | 0
jake_q005 | 0 | 0 | 0
Report saved: evaluation_20261015_090816_703630.json


## 9. Inspect grading disagreements

Start with the largest errors. Read the actual answer and both sets of
dimension-level reasons. Check whether an error came from the model, an ambiguous
rubric, or an incorrect reference mark. Do not assume an AI-prepared reference
is correct merely because it is in the reference file.


In [11]:
answer_by_id = {row["id"]: row for row in answers}
for row in sorted(report["details"], key=lambda item: item["absolute_error"], reverse=True)[:5]:
    print(f"\n{row['answer_id']} | Absolute error: {row['absolute_error']}")
    print(answer_by_id[row["answer_id"]]["answer"])
    print("Reference dimensions:", row["reference_dimension_marks"])
    print("Model dimensions:", row["model_dimension_marks"])



jane_q001 | Absolute error: 1
Nationalism and competition for colonies caused tension in Europe. A Serbian killed an Austrian prince, so Austria went to war with Serbia. Germany and Russia got involved because they supported different sides. I think Britain joined to protect its colonies from Russia. I do not remember the date or how France became involved. The main reason seems to have been the assassination rather than anything about armies.
Reference dimensions: [{'dimension_id': 'd1', 'marks': 2, 'reason': 'Correctly links nationalism and competition for colonies to European tension.'}, {'dimension_id': 'd2', 'marks': 0, 'reason': 'Does not explain militarism or rival alliances and discounts the role of armies.'}, {'dimension_id': 'd3', 'marks': 1, 'reason': "Recognizes assassination of an Austrian prince as the trigger, but omits Franz Ferdinand's name, Sarajevo and date."}, {'dimension_id': 'd4', 'marks': 1, 'reason': "Connects the killing to Austria's war with Serbia, but omits

## 10. Improve one thing and repeat

Tighten one grading instruction, rerun the same 25 answers, and compare MAE.
Keep the answers and references fixed while testing a prompt change. If a
reference mark genuinely needs correction, record a new reviewed reference
version and re-evaluate both runs.

These synthetic student answers provide an easy learning set. Human review
and fresh student work are needed before interpreting the experiment as real
exam-grading performance.
